In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

load_dotenv()
openai = OpenAI()
MODEL = "gpt-4.1-mini"

In [2]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [3]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [4]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [5]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [6]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [7]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [8]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [9]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-4.1-mini
Found 6 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a vibrant and innovative AI community dedicated to building the future of machine learning collaboratively. It serves as a comprehensive platform where researchers, developers, and enterprises create, discover, and share models, datasets, and applications — all designed to accelerate advancements in artificial intelligence.

Founded on the mission to democratize quality machine learning, Hugging Face offers an open and accessible hub encouraging teamwork and innovation "one commit at a time." It boasts a thriving ecosystem that hosts over 2 million machine learning models and more than 500,000 datasets, facilitating a seamless environment for AI exploration and deployment.

---

## What Hugging Face Offers

- **Models:** Explore and deploy over 2 million ready-to-use machine learning models covering a broad array of tasks including NLP, computer vision, audio, reinforcement learning, and more.
- **Datasets:** Access and contribute to more than 500,000 datasets that fuel AI training and evaluation.
- **Spaces:** Run and share AI applications within the Hugging Face community, fostering collaboration and innovation.
- **Community & Collaboration:** Engage with a global community of AI enthusiasts through forums, Discord, and GitHub for knowledge sharing and problem solving.
- **Enterprise Solutions:** Tailored support and infrastructure including Hugging Face PRO, inference endpoints, and storage buckets to help businesses deploy AI solutions efficiently.
- **HuggingChat:** An AI-powered conversation platform integrating state-of-the-art language models.
- **Open Source Stack:** Free and open tools that enable developers to move faster and build powerful AI applications.

---

## Company Culture

At its core, Hugging Face fosters an open and inclusive culture grounded in collaboration and innovation. Emphasizing transparency and community contribution, the company is driven by developers and researchers passionate about making machine learning tools accessible to all.

- **Mission-driven:** Committed to democratizing machine learning with a global community spirit.
- **Collaborative:** A hub for co-creating AI solutions, empowering developers worldwide.
- **Innovative:** Always pushing boundaries with cutting-edge research, open source contributions, and practical AI deployments.
- **Learning & Growth:** Active knowledge sharing through blogs, research papers, tutorials, and community discussions.

---

## Customers & Community

Hugging Face serves a diverse range of users including:

- **Individual Developers & Researchers:** Discover, fine-tune, and share models/datasets to advance AI research and product development.
- **Enterprises & Startups:** Leverage scalable AI infrastructure and expert support to integrate machine learning into business operations.
- **Academic Institutions:** Collaborate and access state-of-the-art AI tools for education and research.
- **Open Source Community:** A strong contributor base that continuously improves and expands Hugging Face’s repositories, making AI technology accessible globally.

The vibrant community is supported by over 185 team members and a massive network of active contributors engaging daily on forums, GitHub, and social platforms.

---

## Careers at Hugging Face

Are you passionate about AI and want to shape the future alongside talented engineers and researchers? Hugging Face offers exciting career opportunities across multiple domains including software engineering, research, machine learning, infrastructure, and customer success.

- **Open Positions:** Check their current openings for roles that suit your skills and aspirations.
- **Work Environment:** Supportive, mission-aligned, and community-focused workplace encouraging growth and innovation.
- **Impact:** Contribute directly to a platform used worldwide by millions in the AI ecosystem.

---

## Learn & Stay Connected

Hugging Face encourages continuous learning with resources including:

- In-depth **Documentation**
- **Tutorials** and **Learning Tracks**
- **Community Blog** featuring articles on NLP, RL, ethics, diffusion models, and more
- Regular updates on latest research papers and AI trends

Join their **Discord** and **Forum** to connect with peers, get support, and contribute to discussions.

---

## Summary

Hugging Face is more than a company — it’s a powerful AI community uniting thousands across the globe to collaboratively build and share machine learning technology for a better future.

Explore their platform to leverage world-class models, datasets, tools, and expertise to accelerate your AI journey.

Visit [huggingface.co](https://huggingface.co) to get started today!

---

*Empowering AI innovation through openness, collaboration, and shared knowledge.*